# 07 · Monitoring and Drift

**Goal:** measure whether today's data still looks like the data the model was trained on, and decide when to alert.

**Three kinds of drift**

| Drift | What changes | Example |
|---|---|---|
| **Data drift** | The inputs | Patients are older on average than in training |
| **Prediction drift** | The model's outputs | The model suddenly flags twice as many people |
| **Concept drift** | The link between inputs and the true answer | A new treatment changes who develops diabetes |

**Three ways to measure data drift** (Lakehouse Monitoring reports the same statistics automatically)

| Statistic | Idea | Rule of thumb |
|---|---|---|
| **KS test** | Largest gap between the two cumulative distributions | p-value < 0.05 → distributions differ |
| **PSI** (population stability index) | How much the share of rows in each bucket moved | < 0.1 stable · 0.1–0.2 moderate · > 0.2 significant |
| **Jensen-Shannon distance** | How different two histograms are, from 0 (same) to 1 | > 0.1 worth a look |

In [ ]:
%run ./Includes/_config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from scipy.spatial.distance import jensenshannon

full = spark.table(TABLE).toPandas()
reference = full.sample(frac=0.5, random_state=11)                     # the "training-time" data
current = full.drop(reference.index).copy()                            # "this month's" data

# Simulate a changing population: older patients and higher BMI in part of this month's data
rng = np.random.default_rng(11)
shifted = rng.random(len(current)) < 0.5
current.loc[shifted, "Age"] = np.minimum(current.loc[shifted, "Age"] + 3, 13)
current.loc[shifted, "BMI"] = current.loc[shifted, "BMI"] + 4
print(f"reference: {len(reference):,} rows   current: {len(current):,} rows")

## 1. Compute drift for every feature

In [ ]:
def psi(ref, cur, bins=10):
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:                                    # binary / low-cardinality column: use its values as buckets
        values = np.union1d(np.unique(ref), np.unique(cur))
        r = np.array([(ref == v).mean() for v in values]); c = np.array([(cur == v).mean() for v in values])
    else:
        r = np.histogram(np.clip(ref, edges[0], edges[-1]), edges)[0] / len(ref)
        c = np.histogram(np.clip(cur, edges[0], edges[-1]), edges)[0] / len(cur)
    r, c = np.clip(r, 1e-6, None), np.clip(c, 1e-6, None)
    return float(np.sum((c - r) * np.log(c / r)))

def js(ref, cur, bins=20):
    edges = np.histogram_bin_edges(np.concatenate([ref, cur]), bins=bins)
    return float(jensenshannon(np.histogram(ref, edges)[0] + 1e-9, np.histogram(cur, edges)[0] + 1e-9))

rows = []
for col in FEATURES:
    ks = ks_2samp(reference[col], current[col])
    rows.append({"feature": col, "ks_statistic": ks.statistic, "ks_p_value": ks.pvalue,
                 "psi": psi(reference[col].values, current[col].values), "js_distance": js(reference[col].values, current[col].values)})
drift = pd.DataFrame(rows)
drift["status"] = np.select([drift["psi"] > 0.2, drift["psi"] > 0.1], ["DRIFT", "watch"], "stable")
drift.sort_values("psi", ascending=False).round(4).head(10)

## 2. See the drift
The two features we changed stand out; the others overlap almost perfectly.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["Age", "BMI", "HighBP"]):
    bins = np.histogram_bin_edges(pd.concat([reference[col], current[col]]), bins=20)
    ax.hist(reference[col], bins=bins, alpha=0.55, density=True, label="reference (training)")
    ax.hist(current[col], bins=bins, alpha=0.55, density=True, label="current")
    status = drift.set_index("feature").loc[col]
    ax.set_title(f"{col}  ·  PSI {status['psi']:.3f}  ·  {status['status']}")
    ax.legend()
plt.tight_layout()
plt.show()

## 3. Prediction drift
Score both datasets with the same model. If the share of positive predictions moves, the model's behaviour has changed
even before any true labels arrive.

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000).fit(reference[FEATURES], reference[LABEL])
ref_rate = model.predict(reference[FEATURES]).mean()
cur_rate = model.predict(current[FEATURES]).mean()
print(f"Predicted diabetes rate  reference: {ref_rate:.1%}   current: {cur_rate:.1%}   change: {cur_rate - ref_rate:+.1%}")

## 4. Turn the numbers into an alert decision

In [ ]:
drifted = drift.loc[drift["status"] == "DRIFT", "feature"].tolist()
prediction_shift = abs(cur_rate - ref_rate)
alert = bool(drifted) or prediction_shift > 0.05
print("Drifted features:", drifted or "none")
print(f"Prediction shift: {prediction_shift:.1%}")
print("ALERT: investigate and consider retraining" if alert else "OK: no action needed")

## 5. The same thing, automated: Lakehouse Monitoring
In production you do not compute this by hand. A **Lakehouse monitor** on the inference table computes profile and drift
metrics on a schedule, builds a dashboard, and SQL alerts fire when a metric crosses a threshold:

```python
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import MonitorInferenceLog, MonitorInferenceLogProblemType
WorkspaceClient().quality_monitors.create(
    table_name=f"{CATALOG}.{SCHEMA}.inference_logs",
    inference_log=MonitorInferenceLog(timestamp_col="timestamp", granularities=["1 day"],
                                      model_id_col="model_version", prediction_col="prediction", label_col="label",
                                      problem_type=MonitorInferenceLogProblemType.PROBLEM_TYPE_CLASSIFICATION),
    baseline_table_name=f"{CATALOG}.{SCHEMA}.baseline",
    output_schema_name=f"{CATALOG}.{SCHEMA}",
    assets_dir=f"/Workspace/Users/{user}/lakehouse_monitoring")
```

The monitor writes a `..._drift_metrics` table with columns such as `ks_test`, `js_distance` and
`population_stability_index`: the same statistics computed above. The full course (A2 · 3.1) builds one end to end.

## Key points
- Compare **current** data with a **reference** (training-time) data set, feature by feature.
- Use more than one statistic: KS for significance, PSI for size of change, JS for histogram difference.
- Watch predictions too: prediction drift is visible before true labels exist.
- Automate it with Lakehouse Monitoring and alerts, then retrain when drift is confirmed.